In [13]:
import pandas as pd
import sqlite3 as sq
import matplotlib as mpl
from matplotlib import rcParams
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import fiona
import os
import io
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.functions import read_census_table, read_wp23_table

In [ ]:
#TN we're reading in from UrbanSIM
tnurbansim = pd.read_csv(r"G:\3-GitHub\FCLUCS_Scenario_Planning\1-data\1-raw\2050RTP-TDMFeedback_2050Block.csv")

In [6]:
col_dict = {'sector_0': 'EMP_OTHER', 'sector_1': 'EMP_SERVICE', 'sector_2': 'EMP_RETAIL', 'sector_3': 'EMP_OFFICE', 'geo_level_id': 'GEO_ID',
               'sector_4': 'EMP_MEDICAL', 'sector_5': 'EMP_INDUSTRIAL', 'sector_6': 'EMP_GOVERNMENT', 'sector_7': 
               'EMP_FOODSERVICE', 'sector_8': 'EMP_EDUCATION', 'household_population': 'HHPOP', 'sum_total_households': 'HH', 'sum_total_jobs': 'EMP'}
tnurbansim = tnurbansim.rename(columns = col_dict)
tnurbansim_filt = tnurbansim[['GEO_ID', 'HHPOP', 'EMP', 'EMP_OTHER', 'EMP_SERVICE', 'EMP_RETAIL', 'EMP_OFFICE', 
'EMP_MEDICAL', 'EMP_INDUSTRIAL', 'EMP_GOVERNMENT', 'EMP_FOODSERVICE', 'EMP_EDUCATION']]

In [8]:
#notably, these aren't rounded to whole jobs. Since we're hexing this I don't think it matters.
tnurbansim_filt.head(2)

,GEO_ID,HHPOP,EMP,EMP_OTHER,EMP_SERVICE,EMP_RETAIL,EMP_OFFICE,EMP_MEDICAL,EMP_INDUSTRIAL,EMP_GOVERNMENT,EMP_FOODSERVICE,EMP_EDUCATION
0,470831203003000,16.582708,2.0,1.0,0.0,0.0,0.0,0.036140,0.035221,0.498547,0.286435,0.030014
1,470831203003090,0.231629,2.0,1.0,0.0,0.0,0.0,0.058606,0.049506,0.517907,0.301769,0.029002


In [9]:
tnurbansim_filt['GEO_ID'] = tnurbansim_filt['GEO_ID'].astype(str)

In [17]:
#KY we will take the same allocations and do 2050 controls, so read in the base year with allocations as well as the WP smoothed totals
ky = gpd.read_file(r"G:\3-GitHub\FCLUCS_Scenario_Planning\2-outputs\2-geospatial\BaseYear.gpkg", layer="KY_HHPOP")
kycontrols = read_wp23_table('DemSmoothed')

In [32]:
kycontrols.head(2)

,NAME,Variable,2023 Census,2023 WP,2024,2025,2026,2027,2028,2029,...,2042,2043,2044,2045,2046,2047,2048,2049,2050,Last Update Pushed_20260812
0,"Cannon County, Tennessee",Population,15063,14854.0,15122.259259,15180.518519,15239.777778,15300.037037,15359.296296,15419.555556,...,16232.925926,16297.185185,16362.444444,16427.703704,16492.962963,16559.222222,16625.481481,16691.740741,16759.0,None
1,"Cheatham County, Tennessee",Population,42254,42068.0,42485.111111,42718.222222,42952.333333,43188.444444,43424.555556,43663.666667,...,46893.111111,47152.222222,47412.333333,47673.444444,47936.555556,48201.666667,48467.777778,48735.888889,49005.0,None


In [34]:
thelist = ['Trigg County, Kentucky', 'Christian County, Kentucky']
kyhorizon = kycontrols[['NAME', '2023 Census', '2050']].loc[kycontrols['NAME'].isin(thelist)]
kyhorizon['HHPOP_GROWTH'] = kyhorizon['2050'] - kyhorizon['2023 Census']

In [35]:
kyhorizon.head(2)

,NAME,2023 Census,2050,HHPOP_GROWTH
17,"Christian County, Kentucky",72032,70260.0,-1772.0
19,"Trigg County, Kentucky",14369,16279.0,1910.0


In [25]:
trigghorizoncontrol = int(kyhorizon.loc[kyhorizon['NAME'] == 'Trigg County, Kentucky', '2050'].iloc[0])
christianhorizoncontrol = int(kyhorizon.loc[kyhorizon['NAME'] == 'Christian County, Kentucky', '2050'].iloc[0])

In [28]:
controls = {'Trigg County, Kentucky': trigghorizoncontrol,'Christian County, Kentucky': christianhorizoncontrol}

ky['HHPOP_2050'] = ((ky['HHPOP_ALLOC'])/100) * ky['COUNTY'].map(controls)

In [29]:
ky.head(2)

,GEO_ID,COUNTY,COUNTYFP,HHPOP,HHPOP_ALLOC,geometry,HHPOP_2050
0,212219701003003,"Trigg County, Kentucky",21221,4,0.028594,"POLYGON ((-87.87697 36.96011, -87.87566 36.959...",4.654800
1,212219701003006,"Trigg County, Kentucky",21221,16,0.114376,"POLYGON ((-87.87628 36.95714, -87.87603 36.957...",18.619201


In [30]:
ky.loc[ky['COUNTY'] == 'Trigg County, Kentucky', 'HHPOP_2050'].sum()

np.float64(16279.0)

In [31]:
trigghorizoncontrol

16279

In [ ]:
#method two would be taking the entire growth and giving the share of growth to the corresponding allocation... 
